第一步：定义 Agent 状态 (State)
状态是 Agent 的“记忆”，记录了当前处理到哪一步、Task ID 是什么、是否有报错。

In [1]:
from typing import Any, Dict, TypedDict, Annotated, List, Optional
import operator
from hipHopProducer import HipHopAutoProject

producer = HipHopAutoProject()
BASE_TEMP_DIR = '/Users/randy/Downloads/temp'
producer.temp_dir = '/Users/randy/Downloads/temp'

class AgentState(TypedDict):
    # 基础信息
    song_name: str
    task_id: Optional[str] = None

    # 专属文件夹路径
    task_dir: str
    
    # 执行进度
    current_step: str
    status: str # pending, processing, completed, failed
    
    # 数据流
    video_path: Optional[str] = None
    video_segs: List[Dict[str, Any]] = []
    video_eng: List[str] = []
    srt_path: Optional[str] = None
    final_mv_path: Optional[str] = None
    
    # 错误记录：记录重试次数或具体报错
    error_log: List[str] = []
    retry_count: int = 0

/Users/randy/Library/Mobile Documents/com~apple~CloudDocs/Document/project/randyTranslation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏗️ 正在初始化模型...


llama_model_load_from_file_impl: using device Metal (Apple M4) - 12124 MiB free
llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /Users/randy/.cache/huggingface/hub/models--Randyliu99--qwen2.5-7b-jcole-gguf/snapshots/797a41ca8b8b3b1335a08e43e3b4f1bb52bb1ae3/./Qwen2.5-7B-Instruct.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Unsloth_Gguf_M7Jqshyv
llama_model_loader: - kv   3:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   4:                         general.size_label str              = 7.6B
llama_model_loader: - kv   5:              

✅ 模型加载完成！


将 FastAPI 封装为 Nodes (节点)
每一个 Node 都是一个函数，负责调用你之前的逻辑并更新状态。

In [ ]:
from langgraph.graph import StateGraph, END
import os

# 定义下载节点
def download_node(state: AgentState):
    # ---------------------------------------------------------------------
    # print(f"--- 节点: 下载视频 [{state['song_name']}] ---")
    # # 调用你现有的 producer.download_step
    # # 这里模拟返回路径
    # tmp_video_path = os.path.join(producer.temp_dir, "raw_video.mp4")
    # path = producer.download_step(state['song_name'],tmp_video_path)
    # return {"video_path": path, "current_step": "download_completed"}
    # ---------------------------------------------------------------------

    # 如果路径已存在且文件有效，跳过（断点续传逻辑）
    save_path = os.path.join(state['task_dir'], "raw_video.mp4")
    if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
        print(f"⏩ 跳过下载: {save_path} 已存在")
        return {"video_path": save_path, "current_step": "download_completed"}

    print(f"📡 正在下载: {state['song_name']} -> {save_path}")
    path = producer.download_step(state['song_name'], save_path)
    return {"video_path": path, "current_step": "download_completed"}

# 定义翻译节点（这里可以加入 Agent 的判断）
def translate_node(state: AgentState):
    print("--- 节点: AI 语义翻译 ---")
    # 这里可以调用你微调的 Qwen
    segs, english_texts = producer.transcribe_step(state['video_path'], os.path.join(state["task_dir"], "temp_audio.wav"))
    return {"video_segs": segs, "video_eng": english_texts, "current_step": "translate_completed"}

def srt_node(state: AgentState):
    print("--- 节点: 生成 SRT ---")
    srt_path = os.path.join(state["task_dir"], "bilingual.srt")
    producer.generate_bilingual_srt(state['video_segs'], state['video_eng'], srt_path)
    return {"srt_path": srt_path, "current_step": "srt_completed"}

def mv_node(state: AgentState):
    print("--- 节点: 生成 MV ---")
    final_mv_path = os.path.join(state["task_dir"], "final_mv.mp4")
    producer.burn_video(state['video_path'], state['srt_path'], final_mv_path)
    return {"final_mv_path": final_mv_path, "current_step": "mv_completed"}

构建逻辑图 (The Graph)
这是 LangGraph 的灵魂，你可以定义条件边（Conditional Edges）。比如：如果下载失败，Agent 会自动尝试修复，而不是报错。

In [ ]:
import sqlite3
from typing import Any, Dict, TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver # 持久化器

workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("downloader", download_node)
workflow.add_node("translator", translate_node)
workflow.add_node("video_mixer", srt_node)
workflow.add_node("mv_generator", mv_node)

# 构建连线
workflow.set_entry_point("downloader")
workflow.add_edge("downloader", "translator")
workflow.add_edge("translator", "video_mixer")
workflow.add_edge("video_mixer", "mv_generator")
workflow.add_edge("mv_generator", END)

# 使用 SQLite 存储进度，库文件叫 'checkpoints.db'
conn = sqlite3.connect("/Users/randy/Downloads/temp/checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)
app = workflow.compile(checkpointer=memory)

In [4]:
import uuid
import os

def run_project(song_name: str, existing_task_id: str = None):
    # 如果传入了 ID，则尝试恢复任务；否则创建新任务
    task_id = existing_task_id or str(uuid.uuid4())[:8]
    task_dir = os.path.join(BASE_TEMP_DIR, task_id)
    os.makedirs(task_dir, exist_ok=True)

    config = {"configurable": {"thread_id": task_id}}

    initial_input = {
        "song_name": song_name,
        "task_id": task_id,
        "task_dir": task_dir,
        "current_step": "init",
        "status": "processing",
        "video_segs": [],
        "video_eng": []
    }

    print(f"任务 ID: {task_id}")
    # 只要 thread_id 一样，app 会自动从数据库寻找上一次跑到的进度
    for event in app.stream(initial_input, config):
        print(f"--- 进度追踪: {list(event.keys())[0]} 完成 ---")


# # 为每个任务创建独立的文件夹，防止多个任务运行时文件冲突
# task_id = str(uuid.uuid4())[:8]
# task_dir = os.path.join("/Users/randy/Downloads/temp", task_id)
# os.makedirs(task_dir, exist_ok=True)

# initial_input = {
#     "song_name": "J. Cole - MIDDLE CHILD",
#     "task_id": task_id,
#     "current_step": "init",
#     "status": "processing"
#     # 其他列表字段会自动按 TypedDict 初始化
# }


# # 初始化输入
# initial_input = {
#     "song_name": "Poor thang",  # 必须提供，因为 download_node 需要它
#     "task_id": "task_20260316_001",      # 建议提供，方便日志追踪
#     "current_step": "start",             # 初始步骤状态
#     "status": "pending",                 # 初始运行状态
#     "video_segs": [],                    # 显式初始化空列表，防止 state 累积干扰
#     "video_eng": [],
#     "error_log": [],
#     "retry_count": 0
# }

# # 启动 Agent
# print("🚀 正在启动嘻哈视频生产线...")
# final_output = app.invoke(initial_input)

# # 查看最终结果
# print(f"✅ 处理完成！成品路径: {final_output['final_mv_path']}")

In [5]:
run_project("Poor thang")

任务 ID: 57360bdc
📡 正在下载: Poor thang -> /Users/randy/Downloads/temp/57360bdc/raw_video.mp4
📥 正在搜索并下载: Poor thang...
[generic] Extracting URL: Poor thang
[youtube:search] Extracting URL: ytsearch1:Poor thang
[download] Downloading playlist: Poor thang
[youtube:search] query "Poor thang": Downloading web client config
[youtube:search] query "Poor thang" page 1: Downloading API JSON
[youtube:search] Playlist Poor thang: Downloading 1 items of 1
[download] Downloading item 1 of 1
[youtube] Extracting URL: https://www.youtube.com/watch?v=3aIjM1YGOO8
[youtube] 3aIjM1YGOO8: Downloading webpage


[youtube] 3aIjM1YGOO8: Downloading android vr player API JSON
[info] 3aIjM1YGOO8: Downloading 1 format(s): 399+140
[download] Destination: /Users/randy/Downloads/temp/57360bdc/raw_video.f399.mp4
[download] 100% of   17.72MiB in 00:00:04 at 4.04MiB/s     
[download] Destination: /Users/randy/Downloads/temp/57360bdc/raw_video.f140.m4a
[download] 100% of    4.48MiB in 00:00:00 at 4.75MiB/s   
[Merger] Merging formats into "/Users/randy/Downloads/temp/57360bdc/raw_video.mp4"
Deleting original file /Users/randy/Downloads/temp/57360bdc/raw_video.f399.mp4 (pass -k to keep)
Deleting original file /Users/randy/Downloads/temp/57360bdc/raw_video.f140.m4a (pass -k to keep)
[download] Finished downloading playlist: Poor thang
--- 进度追踪: downloader 完成 ---
--- 节点: AI 语义翻译 ---
正在提取音轨...


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1 --enable-shared --cc=clang --host-cflags= --host-ldflags= --enable-gpl --enable-libaom --enable-libdav1d --enable-libharfbuzz --enable-libmp3lame --enable-libopus --enable-libsnappy --enable-libsvtav1 --enable-libtheora --enable-libvorbis --enable-libvpx --enable-libx264 --enable-libx265 --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-demuxer=dash --enable-neon --enable-opencl --enable-audiotoolbox --enable-videotoolbox --disable-htmlpages
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/User

🎙️ 正在提取音轨并识别 (Faster-Whisper)...
--- 进度追踪: translator 完成 ---
--- 节点: 生成 SRT ---
正在调用本地 Qwen 模型进行语境翻译...


llama_perf_context_print:        load time =    9538.66 ms
llama_perf_context_print: prompt eval time =    9536.97 ms /  1433 tokens (    6.66 ms per token,   150.26 tokens per second)
llama_perf_context_print:        eval time =   37344.09 ms /   614 runs   (   60.82 ms per token,    16.44 tokens per second)
llama_perf_context_print:       total time =   47635.83 ms /  2047 tokens
llama_perf_context_print:    graphs reused =        594


写入双语字幕到 /Users/randy/Downloads/temp/bilingual.srt...
✅ 所有任务已完成！
--- 进度追踪: video_mixer 完成 ---
--- 节点: 生成 MV ---
🎬 正在使用 FFmpeg 压制成品...

✅ 压制成功！
🎥 成品路径: /Users/randy/Downloads/final_hiphop_mv.mp4


TypeError: Type is not msgpack serializable: numpy.float64